## Creating a historical storm series for CASCADE
Cascade requires an input storm series consisting of the model year the storm occurs, maximum run-up, minimum run-up, wave period, and storm duration. Synthetic storms can be created using the multi-variate sea storm model of Wahl et al. (2016), but this script converts recorded water water level and wave data into a cascade storm input. 

Input files should be csv files with headers. Any missing data will be filled with nans and dropped. Water levels can be pulled from a NOAA tide gauge (https://tidesandcurrents.noaa.gov/) and wave information can be obtained from a WIS Station (https://wisportal.erdc.dren.mil/).

Input files include:
- water levels file: should have datetimes and water levels in m NAVD88
- wave information file: should have datetime, significant wave height (m), and wave period (s) columns. 

Input variables include:
- beach slope (to calculate runup)
- berm elevation in m NAVD88
- conversion from m NAVD88 to m MHW for your tidal gauge (check the NOAA datums for your gauge)

In [1]:
# load necessary packages
import numpy as np
import pandas as pd
from IPython.display import display
import os
from datetime import datetime, timedelta
import rws_waterinfo as rws


In [2]:
# Gets data form RWS pythonw waterinfo wrapper
# Waterlevel at Terschelling Noorzee
params = [
    ["OW", "", "", "WATHTE", "terschelling.noordzee", "1996-01-01", "2022-12-31"]
]
# Retrieve data as Pandas DataFrame
df = rws.get_data(params=params, return_df=True)

# Waves at Eierlandse Gat
wave_params = [
    ["OW", "", "", "Hm0", "eierlandsegat", #Wave height
     "1996-01-01", "2022-12-31"],
    ["OW", "", "", "T_H1/3", "eierlandsegat", #Wave period
     "1996-01-01", "2022-12-31"]
]

wave_df = rws.get_data(params=wave_params, return_df=True)

Downloaded 26/26
Downloaded 52/52


In [3]:
import pandas as pd
import numpy as np

# Keep the storm analysis window in Dutch local time.
start_time = "1996-01-01 00:00:00"
end_time   = "2022-12-31 23:00:00"

# These must be appropriate for the Terschelling site and in m NAP.
# Do not reuse the Wrightsville NAVD88 berm/MHW values without converting
# and validating them for Terschelling.
beach_slope = 0.01       # retain only if appropriate for your beach
berm_elevation = 2    # TODO: berm elevation in m NAP
MHW = 0.82               # TODO: MHW elevation in m NAP

weather_grouping = 24
min_storm_dur = 8
max_storm_dur = 240
save_dfs = True
save_dir = r"C:\Users\6679242\OneDrive - Universiteit Utrecht\Documents\WADWAD\Models\CASCADE\CASCADE_inputs\Terschelling"
save_name = "Terschelling_storms_1996_2022"


def rws_hourly_series(raw, parameter_code, output_name, convert_height_to_m=False):
    """Extract an RWS parameter and return its hourly mean series in UTC."""
    code_columns = [
        col for col in ["Grootheid.Code", "Parameter.Code"]
        if col in raw.columns
    ]
    if not code_columns:
        raise KeyError("No Grootheid.Code or Parameter.Code column found.")

    mask = pd.Series(False, index=raw.index)
    for col in code_columns:
        mask |= raw[col].astype("string").str.upper().eq(parameter_code.upper())

    subset = raw.loc[mask].copy()
    if subset.empty:
        available = {
            col: raw[col].dropna().astype(str).unique()[:30].tolist()
            for col in code_columns
        }
        raise ValueError(
            f"No rows found for {parameter_code}. Available codes: {available}"
        )

    times = pd.to_datetime(subset["Tijdstip"], errors="coerce", utc=True)
    values = pd.to_numeric(
        subset["Meetwaarde.Waarde_Numeriek"], errors="coerce"
    )

    if convert_height_to_m:
        unit_text = (
            subset["Eenheid.Code"].fillna("").astype(str) + " " +
            subset["Eenheid.Omschrijving"].fillna("").astype(str)
        ).str.lower()

        is_cm = unit_text.str.contains(r"\bcm\b|centimeter", regex=True)
        is_m = unit_text.str.contains(r"\bm\b|meter", regex=True)

        if (~(is_cm | is_m)).any():
            unknown_units = unit_text[~(is_cm | is_m)].unique().tolist()
            raise ValueError(
                f"Unrecognized height units for {parameter_code}: "
                f"{unknown_units}"
            )

        values.loc[is_cm] /= 100.0

    series = pd.Series(values.to_numpy(), index=times, name=output_name)
    series = series[~series.index.isna()].sort_index()

    # Restrict to the requested local-time window, expressed in UTC.
    return series.resample("1h").mean()


start_local = pd.Timestamp(start_time, tz="Europe/Amsterdam")
end_local = pd.Timestamp(end_time, tz="Europe/Amsterdam")
start_utc = start_local.tz_convert("UTC")
end_utc = end_local.tz_convert("UTC")

water_level = rws_hourly_series(
    df, "WATHTE", "water_level", convert_height_to_m=True
)
wave_hs = rws_hourly_series(
    wave_df, "Hm0", "Hs", convert_height_to_m=True
)
wave_tp = rws_hourly_series(
    wave_df, "T_H1/3", "Tp", convert_height_to_m=False
)

hourly_index = pd.date_range(start_utc, end_utc, freq="1h", name="Time")
df_merged = pd.concat(
    [water_level, wave_hs, wave_tp], axis=1
).reindex(hourly_index)

print("Hourly rows:", len(df_merged))
print("Missing values by column:\n", df_merged.isna().sum())
print("Actual hourly range:", df_merged.index.min(), "to", df_merged.index.max())

Hourly rows: 236688
Missing values by column:
 water_level      23
Hs             2568
Tp             2568
dtype: int64
Actual hourly range: 1995-12-31 23:00:00+00:00 to 2022-12-31 22:00:00+00:00


<div class="alert alert-block alert-info">
<b>
    <font size="4"> DO NOT EDIT CODE BLOCKS BELOW - all inputs are updated automatically. To change results, please change your inputs in the block above. However, you should review the output below to ensure it makes sense. 
        </font>
    </b>
</div>



### Step 1: Load data and combine into a single dataframe

In [4]:
def find_time_gaps(df, col_name):
    """
    this function finds all values that are NaN and groups them by consecutive time periods
    NOTE: this function does not fill or modify missing data. it just informs you where the data is missing
    df: dataframe
        has datetime indeces and a column with data to evaluate
    col_name: string
        name of col to evaluate in df
    
    """
    
    # check for NaN value and create a datetime list of all corresponding NaNs
    nan_index_numbers = np.where(df[col_name].isnull())[0]  # returns numeric index
    time_list_nans = df.index[nan_index_numbers]  # returns the datetime index

    # loop through the NaN times and check if the difference between times is more than 1 hour
    # if so, that means this is the transition from one consecutive time period of nans to another
    # add the end value plus the next start value to our list of datetimes
    datetime_nans = []
    start = time_list_nans[0]
    datetime_nans.append(start)  # add the first value to the list
    for t in range(len(time_list_nans)-1):
        # if more than 1 hr between this time (t) and the next time in the list, we have transition from one consecutive time period of nans to another
        if time_list_nans[t+1]-time_list_nans[t]!=timedelta(seconds=3600):  
            datetime_nans.append(time_list_nans[t])  # this is the last time step in a consecutive period
            datetime_nans.append(time_list_nans[t+1])  # this is the first time step in the next group of NaNs
    stop = time_list_nans[-1]
    datetime_nans.append(stop)  # add last value to list
    
    # separate out by start and end datetime
    # start dates will be odd indeces, end dates will be even indeces
    start_dt = []
    end_dt = []
    n_days = []
    for t in range(len(datetime_nans)):
        if t%2 == 0:
            start_dt.append(datetime_nans[t])  # just to make sure it always outputs in the Timestamp format
        else:
            end_dt.append(datetime_nans[t])
            
    # create output: missing days and hours to stay separate (e.g. 7 days and 12 hours):
    missing_days = [(end_dt[dt] - start_dt[dt]).days for dt in range(len(start_dt))]
    missing_hours = [(end_dt[dt] - start_dt[dt]).seconds/3600 + 1 for dt in range(len(start_dt))]
    missing_data_ranges = pd.DataFrame(columns=['start date', 'end date', "days", "hours"])
    missing_data_ranges["start date"] = start_dt
    missing_data_ranges["end date"] = end_dt
    missing_data_ranges["days"] = missing_days
    missing_data_ranges["hours"] = missing_hours
    
    print("{0} column is missing the following date ranges".format(col_name))
    display(missing_data_ranges)

In [5]:
import pandas as pd
import numpy as np


def load_data(
    start_time,
    end_time,
    water_df,
    wave_height_df,
    wave_period_df,
    water_code="WATHTE",
    wave_height_code="Hmax",
    wave_period_code="T_H1/3",
    wave_height_column="Hmax",
    timezone="Europe/Amsterdam"
):
    """Combine RWS Waterinfo DataFrames into an hourly dataframe."""

    start_dt = pd.Timestamp(start_time)
    end_dt = pd.Timestamp(end_time)

    if start_dt.tzinfo is None:
        start_dt = start_dt.tz_localize(timezone)
    else:
        start_dt = start_dt.tz_convert(timezone)

    if end_dt.tzinfo is None:
        end_dt = end_dt.tz_localize(timezone)
    else:
        end_dt = end_dt.tz_convert(timezone)

    hourly_index = pd.date_range(
        start=start_dt,
        end=end_dt,
        freq="1h",
        name="Time"
    )

    def extract_hourly(raw, code, output_name, convert_height_to_m=False):
        if "Tijdstip" not in raw.columns:
            raise KeyError("RWS DataFrame must contain a 'Tijdstip' column.")
        if "Meetwaarde.Waarde_Numeriek" not in raw.columns:
            raise KeyError(
                "RWS DataFrame must contain 'Meetwaarde.Waarde_Numeriek'."
            )

        code_columns = [
            col for col in ["Grootheid.Code", "Parameter.Code"]
            if col in raw.columns
        ]
        if not code_columns:
            raise KeyError(
                "RWS DataFrame must contain 'Grootheid.Code' "
                "or 'Parameter.Code'."
            )

        mask = pd.Series(False, index=raw.index)
        for col in code_columns:
            mask |= (
                raw[col].astype("string").str.upper().eq(code.upper())
            )

        subset = raw.loc[mask].copy()
        if subset.empty:
            available = {
                col: raw[col].dropna().astype(str).unique().tolist()
                for col in code_columns
            }
            raise ValueError(
                f"No rows found for {code}. Available codes: {available}"
            )

        times = pd.to_datetime(
            subset["Tijdstip"], errors="coerce", utc=True
        ).dt.tz_convert(timezone)

        values = pd.to_numeric(
            subset["Meetwaarde.Waarde_Numeriek"], errors="coerce"
        )

        # Turn invalid RWS sentinel values into NaN
        values = values.where(np.isfinite(values))
        values = values.mask(values.abs() >= 1e20)

        if convert_height_to_m:
            unit_parts = []
            for col in ["Eenheid.Code", "Eenheid.Omschrijving"]:
                if col in subset.columns:
                    unit_parts.append(
                        subset[col].fillna("").astype(str)
                    )

            if not unit_parts:
                raise KeyError(
                    "Unit metadata is needed to convert wave/water height."
                )

            unit_text = unit_parts[0].str.lower()
            for part in unit_parts[1:]:
                unit_text = unit_text + " " + part.str.lower()

            is_mm = unit_text.str.contains(
                r"\bmm\b|millimeter", regex=True
            )
            is_cm = unit_text.str.contains(
                r"\bcm\b|centimeter", regex=True
            )
            is_m = unit_text.str.contains(
                r"\bm\b|meter", regex=True
            )

            recognized = is_mm | is_cm | is_m
            if (~recognized).any():
                unknown_units = unit_text[~recognized].unique().tolist()
                raise ValueError(
                    f"Unrecognized units for {code}: {unknown_units}"
                )

            values.loc[is_mm] /= 1000.0
            values.loc[is_cm] /= 100.0

        series = pd.Series(
            values.to_numpy(),
            index=pd.DatetimeIndex(times),
            name=output_name
        )
        series = series[~series.index.isna()].dropna()
        series = series.groupby(level=0).mean().sort_index()
        series = series.loc[start_dt:end_dt]

        return series.resample("1h").mean().reindex(hourly_index)

    water_level = extract_hourly(
        water_df, water_code, "water_level",
        convert_height_to_m=True
    )
    wave_height = extract_hourly(
        wave_height_df, wave_height_code, wave_height_column,
        convert_height_to_m=True
    )
    wave_period = extract_hourly(
        wave_period_df, wave_period_code, "Tp",
        convert_height_to_m=False
    )

    df_merged = pd.concat(
        [water_level, wave_height, wave_period], axis=1
    )

    for col in df_merged.columns:
        if df_merged[col].isna().any():
            find_time_gaps(df_merged, col)

    return df_merged.dropna()

In [6]:
df_merged = load_data(
    start_time=start_time,
    end_time=end_time,
    water_df=df,
    wave_height_df=wave_df,
    wave_period_df=wave_df,
    water_code="WATHTE",
    wave_height_code="Hm0",
    wave_period_code="T_H1/3",
    wave_height_column="Hs"
)

water_level column is missing the following date ranges


,start date,end date,days,hours
0,2022-12-31 01:00:00+01:00,2022-12-31 23:00:00+01:00,0,23.0


Hs column is missing the following date ranges


,start date,end date,days,hours
0,1996-11-01 01:00:00+01:00,1996-12-01 00:00:00+01:00,29,24.0
1,1997-02-01 01:00:00+01:00,1997-03-01 00:00:00+01:00,27,24.0
2,1999-10-06 01:00:00+02:00,1999-11-23 00:00:00+01:00,48,1.0
3,2022-12-31 01:00:00+01:00,2022-12-31 23:00:00+01:00,0,23.0


Tp column is missing the following date ranges


,start date,end date,days,hours
0,1996-01-03 13:00:00+01:00,1996-01-03 15:00:00+01:00,0,3.0
1,1996-01-13 19:00:00+01:00,1996-01-13 19:00:00+01:00,0,1.0
2,1996-01-23 07:00:00+01:00,1996-01-23 10:00:00+01:00,0,4.0
3,1996-01-26 22:00:00+01:00,1996-01-26 22:00:00+01:00,0,1.0
4,1996-01-27 07:00:00+01:00,1996-01-27 07:00:00+01:00,0,1.0
...,...,...,...,...
359,2021-01-29 10:00:00+01:00,2021-03-25 10:00:00+01:00,55,1.0
360,2021-04-06 07:00:00+02:00,2021-04-06 07:00:00+02:00,0,1.0
361,2022-01-13 08:00:00+01:00,2022-03-03 09:00:00+01:00,49,2.0
362,2022-07-22 09:00:00+02:00,2022-07-22 14:00:00+02:00,0,6.0


In [7]:
# review merged dataframe
df_merged

,water_level,Hs,Tp
Time,,,
1996-01-01 00:00:00+01:00,-0.510000,1.170000,6.100000
1996-01-01 01:00:00+01:00,-0.031667,1.260000,6.600000
1996-01-01 02:00:00+01:00,0.275000,1.360000,6.400000
1996-01-01 03:00:00+01:00,0.460000,1.570000,6.400000
1996-01-01 04:00:00+01:00,0.511667,1.430000,6.200000
...,...,...,...
2022-12-30 20:00:00+01:00,-1.023333,2.830000,6.950000
2022-12-30 21:00:00+01:00,-0.496667,2.560000,6.366667
2022-12-30 22:00:00+01:00,0.053333,2.383333,6.566667


### Step 2: Calculate the runup to use for the Total Water Level (TWL) 

In [8]:
def calculate_r2_percent(Hs, Tp, slope):
    """Calculate R2% (2% exceedance runup) using Stockdon et al. (2006)"""
    g = 9.81
    # Deep water wavelength
    L0 = (g * Tp**2) / (2 * np.pi)
    
    # Setup component
    setup = 0.35 * slope * np.sqrt(Hs * L0)
    
    # Swash components
    S_incident = 0.75 * slope * np.sqrt(Hs * L0)
    S_infragravity = 0.06 * np.sqrt(Hs * L0)
    S_total = np.sqrt(S_incident**2 + S_infragravity**2)
    
    # R2% runup
    R2 = 1.1 * (setup + S_total/2)
    
    return R2

In [9]:
# calculate R2 based on the beach slope
r2_values = calculate_r2_percent(df_merged['Hs'], df_merged['Tp'], beach_slope)

# add runup and TWL to the combined df
df_merged['R2'] = r2_values
df_merged['TWL'] = df_merged['water_level'] + df_merged['R2']

# review merged dataframe
df_merged

,water_level,Hs,Tp,R2,TWL
Time,,,,,
1996-01-01 00:00:00+01:00,-0.510000,1.170000,6.100000,0.305929,-0.204071
1996-01-01 01:00:00+01:00,-0.031667,1.260000,6.600000,0.343501,0.311834
1996-01-01 02:00:00+01:00,0.275000,1.360000,6.400000,0.346057,0.621057
1996-01-01 03:00:00+01:00,0.460000,1.570000,6.400000,0.371816,0.831816
1996-01-01 04:00:00+01:00,0.511667,1.430000,6.200000,0.343762,0.855429
...,...,...,...,...,...
2022-12-30 20:00:00+01:00,-1.023333,2.830000,6.950000,0.542096,-0.481237
2022-12-30 21:00:00+01:00,-0.496667,2.560000,6.366667,0.472313,-0.024353
2022-12-30 22:00:00+01:00,0.053333,2.383333,6.566667,0.470041,0.523374


### Step 3: Create the CASCADE storms based on berm elevation and TWL

In [10]:
def create_storms(
    df_merged, 
    berm_elevation, 
    weather_grouping=24, 
    MHW=0.421,
    min_storm_dur=8,
    max_storm_dur=240,
    save_dfs=True,
    save_dir="",
    save_name=""
):
    
    """    
    df_merged: dataframe
        output dataframe from the previous function
    berm_elevation: float
        berm elevation of your location of interest [m NAVD88]
    weather_grouping: int, optional
        if storms occur within the specified limit, they are assumed part of same weather system and are grouped into one event [hrs]
    MHW: float, optional
        conversion from m NAVD88 to m MHW for your tidal gauge [m]
    min_storm_dur: int, optional
        minimum storm duration that is considered a storm event [hrs]
    max_storm_dur: int, optional
        maximum duration to include in storm events [hrs]
    save_dfs: bool, optional
        determine whether to save the dataframes as csv and npy files
    savedir: string, optional
        directory to save the storm files. if blank, use the current working directory
    save_name: string, optional
        name of this storm run
    
    """
    # -----------------------------------------------------------------------------
    # ---------------------------STEP 1 IDENTIFY STORMS ---------------------------
    # -----------------------------------------------------------------------------
    # initialize new dataframe
    df = pd.DataFrame({
        "Time": pd.to_datetime(df_merged.index),
        "TWL": pd.to_numeric(df_merged["TWL"], errors="coerce"),
        "Tp": pd.to_numeric(df_merged["Tp"], errors="coerce")
    })
    df = df.sort_values("Time").reset_index(drop=True)

    # determine time steps that are storms (TWL > berm elevation)
    df["AboveBerm"] = df["TWL"] > berm_elevation

    # assign storm start based on consecutive times when TWL > berm elevation
    df["StormStart"] = df["AboveBerm"] & (~df["AboveBerm"].shift(1, fill_value=False)) 

    # adjust StormStart for continuous weather system
    temp_df = df[df["AboveBerm"]]
    index_vals = temp_df.index
    for i in range(1, len(index_vals)):  # skip the first row
        current_row = temp_df.loc[index_vals[i]]
        prev_row = temp_df.loc[index_vals[i-1]]
        if current_row.StormStart==True:  # if storm start is set to true, check if it is within limit hours of the previous False
            current_time = current_row.Time
            prev_row_time = prev_row.Time
            time_delta_hrs = (current_time - prev_row_time).days*24 + (current_time - prev_row_time).seconds/3600
            if time_delta_hrs < weather_grouping:  # if it is within limit, we want to make it part of the previous storm instead of a new storm
                df.loc[index_vals[i], "StormStart"] = False  # index values should be the same in the main df

    df["StormID"] = df["StormStart"].cumsum()
    df.loc[~df["AboveBerm"], "StormID"] = np.nan
    
    
    # -----------------------------------------------------------------------------
    # ---------------------------STEP 2 CREATE STORMS ---------------------------
    # -----------------------------------------------------------------------------
    storms = []
    storm_groups = df.dropna(subset=["StormID"]).groupby("StormID")

    for sid, group in storm_groups:

        group = group.sort_values("Time").copy()
        start_time = group["Time"].iloc[0]
        end_time   = group["Time"].iloc[-1]

        # Duration (hours)
        duration = len(group)  # each row is 1 hour where an exceedance occured

        # Rhigh and Rlow from TWL during the storm (in m NAVD88)
        rhigh = group["TWL"].max()
        rlow  = group["TWL"].min()
        # convert to decameters MHW
        rhigh = (rhigh - MHW) / 10
        rlow = (rlow - MHW) / 10

        # Period from Tp at peak TWL
        peak_idx = group["TWL"].idxmax()
        period = group.loc[peak_idx, "Tp"]
        

        # only add storms > specified duration (Magliocca et al., 2011) but less than maximum 
        if duration >= min_storm_dur and duration <= max_storm_dur:
            storms.append({
                "calendar_year": start_time.year,
                "StartTime": start_time,
                "EndTime": end_time,
                "Rhigh": rhigh,
                "Rlow": rlow,
                "period": period,
                "duration": duration
            })

    storms_df = pd.DataFrame(storms)

    # =========================
    # CONVERT YEAR → TIME (CASCADE)
    # =========================
    if not storms_df.empty:
        start_year = storms_df["calendar_year"].min()
        storms_df["time"] = storms_df["calendar_year"] - start_year + 1
    else:
        storms_df["time"] = pd.Series(dtype=float)

    # =========================
    # FINAL FORMAT
    # =========================
    cascade_df = storms_df[["time", "Rhigh", "Rlow", "period", "duration"]].copy()
    cascade_df = cascade_df.reset_index(drop=True)
    casc_input_array = cascade_df.to_numpy()
    total_storms = len(cascade_df)
    avg_duration = round(np.mean(cascade_df.duration.values),0)

    # =========================
    # SAVE
    # =========================
    if save_dir == "":
        save_dir = os.getcwd()
    
    if save_name == "":    
        save_storm = os.path.join(save_dir, "storm_summary.csv")
        save_casc = os.path.join(save_dir, "cascade_storms.csv")
        save_npy = os.path.join(save_dir, "cascade_storms.npy")
    else:
        save_storm = os.path.join(save_dir, "{0}_summary.csv".format(save_name))
        save_casc = os.path.join(save_dir, "{0}.csv".format(save_name))
        save_npy = os.path.join(save_dir, "{0}.npy".format(save_name))
    
    if save_dfs:
        storms_df.to_csv(save_storm, index=False)
        cascade_df.to_csv(save_casc, index=False)
        np.save(save_npy, casc_input_array)
        print("dataframes saved to {0}".format(save_dir))
    else:
        print("Preview of storms below. Set save_dfs to True to save full versions.")
        print("Total storms: {0}".format(total_storms))
        print("Average duration: {0} hours".format(avg_duration))
        print("\nStorm summary:")
        display(storms_df.head())

        print("\nCASCADE CSV format (will not have index column):")
        display(cascade_df.head(5))

        print("\nCASCADE NPY format:")
        display(casc_input_array[0:5, :])


In [11]:
# Check that the loader produced the required input columns
print(df_merged.columns.tolist())

# Calculate run-up and total water level
df_merged["R2"] = calculate_r2_percent(
    df_merged["Hs"],
    df_merged["Tp"],
    beach_slope
)
df_merged["TWL"] = df_merged["water_level"] + df_merged["R2"]

df_merged.index.name = None

# Now run storm identification
create_storms(
    df_merged=df_merged,
    berm_elevation=berm_elevation,
    weather_grouping=weather_grouping,
    MHW=MHW,
    min_storm_dur=min_storm_dur,
    max_storm_dur=max_storm_dur,
    save_dfs=save_dfs,
    save_dir=save_dir,
    save_name=save_name
)

['water_level', 'Hs', 'Tp', 'R2', 'TWL']
dataframes saved to C:\Users\6679242\OneDrive - Universiteit Utrecht\Documents\WADWAD\Models\CASCADE\CASCADE_inputs\Terschelling
